# Capítulo 7 – Explicabilidade e Interpretação

> Este notebook cobre mapas de atenção, SHAP e Grad-CAM adaptado para espectrogramas.

In [ ]:
# (Opcional) Instalar dependências em um ambiente local
# !pip install torch torchaudio transformers librosa shap matplotlib scikit-learn xgboost

## 7.1 Mapas de Atenção (Wav2Vec2)

In [ ]:
from transformers import Wav2Vec2Processor, Wav2Vec2Model
import torch, librosa, matplotlib.pyplot as plt

processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")
model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")
model.eval()

# y, sr = librosa.load("voz.wav", sr=16000)  # descomente e forneça um arquivo válido
# inputs = processor(y, sampling_rate=sr, return_tensors="pt", padding=True)
# with torch.no_grad():
#     outputs = model(**inputs, output_attentions=True)
# attentions = outputs.attentions  # lista [camadas] de tensores (B, H, T, T)

# Exemplo fictício de visualização (substitua por 'attentions' reais):
import numpy as np
att = np.random.rand(64,64)
plt.figure(figsize=(5,4))
plt.imshow(att, aspect='auto', origin='lower')
plt.title("Mapa de atenção – exemplo")
plt.xlabel("Passo (chave)"); plt.ylabel("Passo (query)")
plt.colorbar(label="peso"); plt.tight_layout(); plt.show()

## 7.2 SHAP – impacto das features em classificador

In [ ]:
import xgboost as xgb, shap, numpy as np
X_emb = np.random.normal(0,1,(300,64))
y = (np.random.rand(300) > 0.5).astype(int)
clf = xgb.XGBClassifier(n_estimators=150, max_depth=4, learning_rate=0.05, random_state=42).fit(X_emb, y)
explainer = shap.TreeExplainer(clf)
shap_values = explainer(X_emb)
shap.plots.beeswarm(shap_values, max_display=20)

## 7.3 Grad-CAM sobre mel-espectrograma (CNN simples)

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, matplotlib.pyplot as plt

class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(32, 1)
    def forward(self, x):
        x = F.relu(self.conv1(x))
        self.feat = F.relu(self.conv2(x))
        x = self.pool(self.feat).squeeze(-1).squeeze(-1)
        return self.fc(x)

model = TinyCNN().eval()

# S_db fictício para demonstração (substitua por seu mel db real)
S_db = np.random.randn(128, 200).astype(np.float32)
X = torch.tensor(S_db[None, None, ...])

logit = model(X).squeeze()
model.zero_grad(); logit.backward(retain_graph=True)
model.feat.retain_grad()
grads = model.feat.grad

weights = grads.mean(dim=(2,3), keepdim=True)
cam = (weights * model.feat).sum(dim=1, keepdim=True)
cam = F.relu(cam).squeeze().detach().numpy()
cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

plt.figure(figsize=(8,3))
plt.imshow(S_db, aspect='auto', origin='lower', cmap='gray')
plt.imshow(cam, aspect='auto', origin='lower', alpha=0.45)
plt.title('Grad-CAM sobre mel-espectrograma (demo)')
plt.xlabel('Tempo'); plt.ylabel('Mel-bandas'); plt.tight_layout(); plt.show()